# Kenya Time-Series Forecasting
**AquaML – Pillar 1**

Convert continuous **stage (water level)** from logger stations on the Mukurumudzi and Ramisi rivers into **discharge** using rating curves, then forecast discharge.

**Stations (Gro for GooD / coastal Kenya)**
- `3KD06` – Mukurumudzi River
- `KISCOL_Weir` – Mukurumudzi River
- `Eshu_Bridge` – Ramisi River

**Pipeline**
1. Load stage time series
2. Fit / apply rating curve (stage to discharge)
3. Aggregate to daily discharge
4. Forecast with lag features + XGBoost
5. Evaluate with NSE, KGE, RMSE


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

from src.evaluation import regression_report
from src.preprocessing import create_lag_features, temporal_train_test_split
from src.visualization import plot_timeseries_forecast, plot_residuals

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.dpi"] = 120


## 1. Load rating curves and stage series

In [ ]:
rc_dir = ROOT / "data/kenya/rating_curves"
ts_dir = ROOT / "data/kenya/timeseries"

stations = pd.read_csv(rc_dir / "stations_metadata.csv")
print(stations)

rc = pd.read_csv(rc_dir / "rating_curves_all.csv")
print("\nRating curve rows:", len(rc))
print(rc.groupby(["station_id", "source"]).size())


In [ ]:
stage = pd.read_csv(ts_dir / "mukurumudzi_3KD06_water_level.csv", parse_dates=["datetime"])
stage = stage.sort_values("datetime").dropna(subset=["stage_m"])
print(stage.head())
print("\nPeriod:", stage["datetime"].min(), "->", stage["datetime"].max())
print("Records:", len(stage))


## 2. Build stage to discharge converter from measured rating points

Fit a power-law rating curve on the **measured** H-Q pairs, then apply it to the logger stage series.

In [ ]:
meas = rc[(rc["station_id"] == "3KD06") & (rc["source"] == "measured")].copy()
meas = meas.dropna(subset=["stage_m", "discharge_m3s"])
meas = meas[meas["discharge_m3s"] > 0].sort_values("stage_m")
print("Measured H-Q points:", len(meas))
print(meas.head())

def power_law(h, a, h0, b):
    return a * np.maximum(h - h0, 0) ** b

p0 = [5.0, float(meas["stage_m"].min()) - 0.05, 1.8]
try:
    popt, _ = curve_fit(power_law, meas["stage_m"].values, meas["discharge_m3s"].values,
                        p0=p0, maxfev=5000, bounds=([0, -1, 0.5], [200, 2, 5]))
    a, h0, b = popt
    print(f"Fitted: Q = {a:.3f} * (H - {h0:.3f})^{b:.3f}")
except Exception as e:
    print("Fit failed, using interpolation only:", e)
    a, h0, b = None, None, None


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(meas["stage_m"], meas["discharge_m3s"], c="#1f77b4", label="Measured", zorder=3)
if a is not None:
    h_line = np.linspace(meas["stage_m"].min(), meas["stage_m"].max() + 0.3, 100)
    ax.plot(h_line, power_law(h_line, a, h0, b), "r-", label="Power-law fit")
sim = rc[(rc["station_id"] == "3KD06") & (rc["source"] == "hec_ras_simulated")]
ax.scatter(sim["stage_m"], sim["discharge_m3s"], c="#ff7f0e", s=20, alpha=0.6, label="HEC-RAS sim")
ax.set_xlabel("Stage H (m)")
ax.set_ylabel("Discharge Q (m3/s)")
ax.set_title("3KD06 Rating Curve")
ax.legend()
plt.tight_layout()
plt.show()


## 3. Apply rating curve to logger stage -> daily discharge

Logger stage may sit on a different datum than the rating-curve staff gauge. We estimate an offset so that median logger stage maps near median rating stage, then interpolate.

In [ ]:
print("Logger stage range:", round(stage["stage_m"].min(), 3), "->", round(stage["stage_m"].max(), 3))
print("Rating stage range:", round(meas["stage_m"].min(), 3), "->", round(meas["stage_m"].max(), 3))

f_q = interp1d(meas["stage_m"].values, meas["discharge_m3s"].values,
               kind="linear", bounds_error=False, fill_value=np.nan)

logger_med = stage["stage_m"].median()
rating_med = meas["stage_m"].median()
offset = logger_med - rating_med
print(f"Suggested datum offset (logger - rating): {offset:.3f} m")

stage["stage_aligned"] = stage["stage_m"] - offset
stage["discharge_m3s"] = f_q(stage["stage_aligned"])
print("Discharge non-null after mapping:", int(stage["discharge_m3s"].notna().sum()))


In [ ]:
daily = (stage.set_index("datetime")
         .resample("D")
         .agg(stage_m=("stage_m", "mean"),
              discharge_m3s=("discharge_m3s", "mean"),
              n_obs=("stage_m", "count"))
         .dropna(subset=["discharge_m3s"])
         .reset_index())
print(daily.head())
print("Daily records:", len(daily))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(daily["datetime"], daily["stage_m"], color="#1f77b4", linewidth=0.8)
axes[0].set_ylabel("Stage (m)")
axes[0].set_title("3KD06 – Daily mean stage and derived discharge")
axes[1].plot(daily["datetime"], daily["discharge_m3s"], color="#2ca02c", linewidth=0.8)
axes[1].set_ylabel("Discharge (m3/s)")
plt.tight_layout()
plt.show()


## 4. Forecast daily discharge (XGBoost + lags)

In [ ]:
df = daily[["datetime", "discharge_m3s"]].dropna().copy()
df = create_lag_features(df, "discharge_m3s", lags=[1, 2, 3, 7, 14])
train_df, test_df = temporal_train_test_split(df, test_ratio=0.25, date_col="datetime")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

feature_cols = [c for c in train_df.columns if c not in ["datetime", "discharge_m3s"]]
X_train, y_train = train_df[feature_cols].values, train_df["discharge_m3s"].values
X_test, y_test = test_df[feature_cols].values, test_df["discharge_m3s"].values

y_persist = test_df["discharge_m3s_lag1"].values
print("Persistence:", regression_report(y_test, y_persist, prefix="Persist_"))

xgb = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
y_xgb = xgb.predict(X_test)
print("XGBoost:", regression_report(y_test, y_xgb, prefix="XGB_"))


In [ ]:
plot_timeseries_forecast(
    test_df["datetime"].values, y_test, y_xgb,
    title="3KD06 Daily Discharge Forecast (XGBoost)",
    ylabel="Discharge (m3/s)"
)
plt.show()
plot_residuals(y_test, y_xgb, title="XGBoost Residuals – 3KD06")
plt.show()


## 5. Other stations available

Same workflow applies to KISCOL and Eshu Bridge.

In [ ]:
kis = pd.read_csv(ts_dir / "mukurumudzi_KISCOL_water_level.csv", parse_dates=["datetime"])
print("KISCOL records:", len(kis), "| columns:", list(kis.columns))
eshu = pd.read_csv(ts_dir / "ramisi_eshu_logger_water_level.csv", parse_dates=["datetime"])
print("Eshu logger records:", len(eshu))


## Takeaways – Time-Series Pillar

- Real Kenyan logger stage + published rating curves produce physically meaningful discharge.
- Temporal split + lag features + XGBoost give a strong, interpretable baseline.
- Always report **NSE** and **KGE** (hydrology standard), not only RMSE/R2.
- Upgrades: multi-horizon forecast, rainfall exogenous input, LSTM / TFT, uncertainty bands.

**Data used**
- `data/kenya/timeseries/mukurumudzi_3KD06_water_level.csv`
- `data/kenya/rating_curves/rating_curve_3KD06.csv`
